# MetAgeFormer — 1. Embedding Extraction

Load the pretrained NMR backbone and extract metabolomic sample embeddings.

**Requirements**
- Released weights under `Model_Weights/MetAgeFormer/` (`config.json`, `tokenizer.pkl`, `model_weights.pth`)
- An AnnData NMR dataset with layer `Z-score normalized` and `var_names` matching the tokenizer vocabulary

No restricted data? Generate a synthetic demo dataset first:
```bash
python Data/generate_fake_data.py --synthetic --outdir Data/fake
```
and point `DATA_PATH` below at `Data/fake/NMR_dataset_fake/val.h5ad`.

In [ ]:
import json
import sys
from pathlib import Path

# Resolve the repo root (works whether the notebook is run from the repo root or Notebooks/)
REPO_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "Src"))

import numpy as np
import pandas as pd
import torch
import anndata as ad

from utils import load_tokenizer
from metageformer_torch.models import MetAgeFormer_Pretrained

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Load the pretrained backbone

The checkpoint layout follows `Model_Weights/readme.md`. The released `config.json` contains
all backbone architecture fields; only `average_attn_weights` is overridden here so per-head
attention scores are available.

In [ ]:
WEIGHTS_DIR = REPO_ROOT / "Model_Weights" / "MetAgeFormer"
tokenizer_path = WEIGHTS_DIR / "tokenizer.pkl"
config_path = WEIGHTS_DIR / "config.json"
weights_path = WEIGHTS_DIR / "model_weights.pth"

for path in (tokenizer_path, config_path, weights_path):
    assert path.exists(), f"Missing {path} — download released weights first (Model_Weights/readme.md)"

tokenizer = load_tokenizer(str(tokenizer_path))
with open(config_path) as f:
    model_config = json.load(f)
model_config["average_attn_weights"] = False  # keep per-head attention for analysis

embedding_module_conf = {"n_vocabs": {"identifier": tokenizer.vocab_size_identifiers}}
model = MetAgeFormer_Pretrained(embedding_module_conf, model_config, str(weights_path))
model.to(DEVICE).eval()
print("Loaded MetAgeFormer pretrained backbone")

## 2. Input data (AnnData NMR)

Required: `adata.layers['Z-score normalized']` and `var_names` that are all in the tokenizer
vocabulary (107 non-derived NMR measures for the released checkpoint).

In [ ]:
DATA_PATH = REPO_ROOT / "Data" / "NMR_dataset_fullcohort_107nonderived" / "val.h5ad"
# Fake-data demo: DATA_PATH = REPO_ROOT / "Data" / "fake" / "NMR_dataset_fake" / "val.h5ad"

adata = ad.read_h5ad(DATA_PATH)
adata

## 3. Tokenize and extract embeddings

`masking='missing'` masks all NaN (missing) measurements with the learned mask token —
no zero-filling. The CLS token output is the sample embedding.

In [ ]:
data_layer = "Z-score normalized"
batch = adata[:64]  # demo subset

inputs, _ = tokenizer.tokenize_from_anndata(
    batch,
    padding="longest",
    masking="missing",
    data_layer=data_layer,
    mode="inference",
    return_tensor=True,
    device=DEVICE,
)

with torch.inference_mode():
    outputs = model(inputs)

embs = outputs["embs"].cpu().numpy()
print("Embedding matrix shape:", embs.shape)  # (n_samples, d_model)

## 4. Save and inspect the embeddings

In [ ]:
emb_df = pd.DataFrame(embs, index=batch.obs_names)
out_csv = REPO_ROOT / "Data" / "demo_embeddings.csv"
emb_df.to_csv(out_csv)
print("saved:", out_csv)
emb_df.head()

## 5. (Optional) First two principal components

Uses numpy SVD only — no extra dependencies. `matplotlib` is optional (`pip install matplotlib`).

In [ ]:
try:
    import matplotlib.pyplot as plt

    x_centered = embs - embs.mean(axis=0)
    u, s, _ = np.linalg.svd(x_centered, full_matrices=False)
    pc = u[:, :2] * s[:2]

    plt.figure(figsize=(5, 4))
    plt.scatter(pc[:, 0], pc[:, 1], s=12)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Sample embeddings (first 2 PCs)")
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping plot")